# Breast Cancer Wisconsin classification using a 1D CNN

This notebook follows the same structure as your SVM notebook, but replaces the `SVC(kernel="linear")` model with a small **1D Convolutional Neural Network**.

The important difference is that CNNs expect input with an extra shape dimension.  
Your original table has this shape:

```python
(samples, features)
```

For a 1D CNN, we reshape it to:

```python
(samples, features, 1)
```

This treats each row of features like a short 1D sequence.

## Cell 1 — Imports

This imports the same tools as before, plus:

- `StandardScaler` to scale the feature values before training the neural network.
- `tensorflow` / `keras` to build the CNN.
- `numpy` for reshaping and prediction handling.

In [1]:
import re
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Cell 2 — Read the `.names` file

This is the same as before.  
It reads the file that contains the column descriptions.

In [2]:
# Read the .names file
with open("breast-cancer-wisconsin.names", "r") as file:
    names_text = file.read()

# Extract only the Attribute Information section
attribute_section = names_text.split("7. Attribute Information")[1]

## Cell 3 — Extract the column names

This is also the same as before.  
It finds the attribute names in the `.names` file, cleans them, and prints the result.

In [3]:
# Extract names from lines like:
# 1. Sample code number            id number
# 11. Class:                       (2 for benign, 4 for malignant)
matches = re.findall(
    r"^\s*\d+\.\s+(.+?)(?:\s{2,}|:)",
    attribute_section,
    flags=re.MULTILINE
)

# Clean names
columns = [
    name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    for name in matches
]

print(columns)
print("Number of columns:", len(columns))

['sample_code_number', 'clump_thickness', 'uniformity_of_cell_size', 'uniformity_of_cell_shape', 'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei', 'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class', 'missing_attribute_values', 'class_distribution']
Number of columns: 13


## Cell 4 — Load the dataset

This is the same as before.

The `.data` file uses `?` for missing values, so `na_values="?"` tells pandas to treat these as missing values.

In [4]:
columns = columns[:11]

df = pd.read_csv(
    "breast-cancer-wisconsin.data",
    header=None,
    names=columns,
    na_values="?"
)

print(df.isna().sum())

df.head()

sample_code_number              0
clump_thickness                 0
uniformity_of_cell_size         0
uniformity_of_cell_shape        0
marginal_adhesion               0
single_epithelial_cell_size     0
bare_nuclei                    16
bland_chromatin                 0
normal_nucleoli                 0
mitoses                         0
class                           0
dtype: int64


,sample_code_number,clump_thickness,uniformity_of_cell_size,uniformity_of_cell_shape,marginal_adhesion,single_epithelial_cell_size,bare_nuclei,bland_chromatin,normal_nucleoli,mitoses,class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


## Cell 5 — Check data types

This checks whether the columns are numeric or object/string types.

In [5]:
print(df.dtypes)

sample_code_number               int64
clump_thickness                  int64
uniformity_of_cell_size          int64
uniformity_of_cell_shape         int64
marginal_adhesion                int64
single_epithelial_cell_size      int64
bare_nuclei                    float64
bland_chromatin                  int64
normal_nucleoli                  int64
mitoses                          int64
class                            int64
dtype: object


## Cell 6 — Check missing values before cleaning

This is the same check as before, shown clearly before we fill missing values.

In [6]:
print("Missing values before cleaning:")
print(df.isna().sum())

Missing values before cleaning:
sample_code_number              0
clump_thickness                 0
uniformity_of_cell_size         0
uniformity_of_cell_shape        0
marginal_adhesion               0
single_epithelial_cell_size     0
bare_nuclei                    16
bland_chromatin                 0
normal_nucleoli                 0
mitoses                         0
class                           0
dtype: int64


## Cell 7 — Fill missing values

This fills missing `bare_nuclei` values using the mode, which means the most common value.

This is the same approach as your SVM version.

In [7]:
# Fill missing bare_nuclei values with the mode
mode_value = df["bare_nuclei"].mode()[0]
df["bare_nuclei"] = df["bare_nuclei"].fillna(mode_value)

print("Missing values after cleaning:")
print(df.isna().sum())

Missing values after cleaning:
sample_code_number             0
clump_thickness                0
uniformity_of_cell_size        0
uniformity_of_cell_shape       0
marginal_adhesion              0
single_epithelial_cell_size    0
bare_nuclei                    0
bland_chromatin                0
normal_nucleoli                0
mitoses                        0
class                          0
dtype: int64


## Cell 8 — Convert columns to integers

This is the same as before.

After this, the whole dataframe is numeric.

In [8]:
# Convert all columns to numeric
df = df.astype(int)

print(df.dtypes)

sample_code_number             int64
clump_thickness                int64
uniformity_of_cell_size        int64
uniformity_of_cell_shape       int64
marginal_adhesion              int64
single_epithelial_cell_size    int64
bare_nuclei                    int64
bland_chromatin                int64
normal_nucleoli                int64
mitoses                        int64
class                          int64
dtype: object


## Cell 9 — Create X and y

This part is slightly different from the SVM version.

The original class labels are:

- `2` = benign
- `4` = malignant

For the CNN, we convert them to:

- `0` = benign
- `1` = malignant

This makes the output layer simpler because the CNN can use one sigmoid neuron for binary classification.

In [9]:
# X = features, y = target
X = df.drop(columns=["sample_code_number", "class"])

# Convert class labels:
# 2 becomes 0
# 4 becomes 1
y = df["class"].map({2: 0, 4: 1})

print("Feature shape:", X.shape)
print("Target values:")
print(y.value_counts())

Feature shape: (699, 9)
Target values:
class
0    458
1    241
Name: count, dtype: int64


## Cell 10 — CNN model function

This is new.

Instead of creating `SVC(kernel="linear")`, we create a small 1D CNN.

The model uses:

- `Conv1D` to learn small patterns across the feature list.
- `Flatten` to turn CNN outputs into one vector.
- `Dense` layers for classification.
- `sigmoid` output because this is binary classification.

In [10]:
def build_cnn_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=2,
            activation="relu"
        ),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            16,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

## Cell 11 — Fixed split using `random_state=42`

This is the CNN version of your fixed SVM split.

Important differences:

1. The features are scaled using `StandardScaler`.
2. The data is reshaped from `(samples, features)` to `(samples, features, 1)`.
3. The CNN is trained using `model.fit()`.
4. The CNN outputs probabilities, so we convert them to `0` or `1` using a threshold of `0.5`.

In [11]:
# Fixed split using random_state=42

# Make the neural network more repeatable
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

X_train_42, X_test_42, y_train_42, y_test_42 = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Fixed split with random_state=42:")
print("Training set:", X_train_42.shape)
print("Test set:", X_test_42.shape)

# Scale the data
scaler_42 = StandardScaler()

X_train_42_scaled = scaler_42.fit_transform(X_train_42)
X_test_42_scaled = scaler_42.transform(X_test_42)

# Reshape for CNN:
# from (samples, features)
# to   (samples, features, 1)
X_train_42_cnn = X_train_42_scaled.reshape(X_train_42_scaled.shape[0], X_train_42_scaled.shape[1], 1)
X_test_42_cnn = X_test_42_scaled.reshape(X_test_42_scaled.shape[0], X_test_42_scaled.shape[1], 1)

print("CNN training shape:", X_train_42_cnn.shape)
print("CNN test shape:", X_test_42_cnn.shape)

# Build and train the CNN
model_42 = build_cnn_model(input_shape=(X_train_42_cnn.shape[1], 1))

history_42 = model_42.fit(
    X_train_42_cnn,
    y_train_42,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)

# Predict probabilities
y_prob_42 = model_42.predict(X_test_42_cnn)

# Convert probabilities to class labels
y_pred_42 = (y_prob_42 >= 0.5).astype(int).ravel()

print("\nAccuracy:", accuracy_score(y_test_42, y_pred_42))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_42, y_pred_42))

print("\nClassification Report:")
print(classification_report(
    y_test_42,
    y_pred_42,
    target_names=["benign", "malignant"]
))

Fixed split with random_state=42:
Training set: (559, 9)
Test set: (140, 9)
CNN training shape: (559, 9, 1)
CNN test shape: (140, 9, 1)
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

Accuracy: 0.9642857142857143

Confusion Matrix:
[[89  3]
 [ 2 46]]

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.97      0.97        92
   malignant       0.94      0.96      0.95        48

    accuracy                           0.96       140
   macro avg       0.96      0.96      0.96       140
weighted avg       0.96      0.96      0.96       140



## Cell 12 — F1 score table for the fixed split

This gives the F1-score table from the classification report.

The F1-score combines precision and recall into one score.

In [12]:
report_42 = classification_report(
    y_test_42,
    y_pred_42,
    target_names=["benign", "malignant"],
    output_dict=True
)

f1_table_42 = pd.DataFrame(report_42).transpose()

print("F1-score table for fixed split:")
f1_table_42[["precision", "recall", "f1-score", "support"]]

F1-score table for fixed split:


,precision,recall,f1-score,support
benign,0.978022,0.967391,0.972678,92.000000
malignant,0.938776,0.958333,0.948454,48.000000
accuracy,0.964286,0.964286,0.964286,0.964286
macro avg,0.958399,0.962862,0.960566,140.000000
weighted avg,0.964566,0.964286,0.964372,140.000000


## Cell 13 — True random split each time

This is the CNN version of your random split.

The key difference is that there is **no `random_state`**, so the train/test split can change each time you run the cell.

The CNN training can also vary slightly because neural networks use randomness during training.

In [13]:
# True random split each time

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y
)

print("Random split:")
print("Training set:", X_train_random.shape)
print("Test set:", X_test_random.shape)

# Scale the data
scaler_random = StandardScaler()

X_train_random_scaled = scaler_random.fit_transform(X_train_random)
X_test_random_scaled = scaler_random.transform(X_test_random)

# Reshape for CNN
X_train_random_cnn = X_train_random_scaled.reshape(
    X_train_random_scaled.shape[0],
    X_train_random_scaled.shape[1],
    1
)

X_test_random_cnn = X_test_random_scaled.reshape(
    X_test_random_scaled.shape[0],
    X_test_random_scaled.shape[1],
    1
)

print("CNN training shape:", X_train_random_cnn.shape)
print("CNN test shape:", X_test_random_cnn.shape)

# Build and train the CNN
model_random = build_cnn_model(input_shape=(X_train_random_cnn.shape[1], 1))

history_random = model_random.fit(
    X_train_random_cnn,
    y_train_random,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)

# Predict probabilities
y_prob_random = model_random.predict(X_test_random_cnn)

# Convert probabilities to class labels
y_pred_random = (y_prob_random >= 0.5).astype(int).ravel()

print("\nAccuracy:", accuracy_score(y_test_random, y_pred_random))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_random, y_pred_random))

print("\nClassification Report:")
print(classification_report(
    y_test_random,
    y_pred_random,
    target_names=["benign", "malignant"]
))

Random split:
Training set: (559, 9)
Test set: (140, 9)
CNN training shape: (559, 9, 1)
CNN test shape: (140, 9, 1)
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

Accuracy: 0.9642857142857143

Confusion Matrix:
[[89  3]
 [ 2 46]]

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.97      0.97        92
   malignant       0.94      0.96      0.95        48

    accuracy                           0.96       140
   macro avg       0.96      0.96      0.96       140
weighted avg       0.96      0.96      0.96       140



## Cell 14 — F1 score table for the random split

This prints the F1-score table for the true random split.

In [14]:
report_random = classification_report(
    y_test_random,
    y_pred_random,
    target_names=["benign", "malignant"],
    output_dict=True
)

f1_table_random = pd.DataFrame(report_random).transpose()

print("F1-score table for random split:")
f1_table_random[["precision", "recall", "f1-score", "support"]]

F1-score table for random split:


,precision,recall,f1-score,support
benign,0.978022,0.967391,0.972678,92.000000
malignant,0.938776,0.958333,0.948454,48.000000
accuracy,0.964286,0.964286,0.964286,0.964286
macro avg,0.958399,0.962862,0.960566,140.000000
weighted avg,0.964566,0.964286,0.964372,140.000000


## Cell 15 — Compare fixed split and random split

This combines the F1-scores from both runs into one table.

This is useful because the random split may perform slightly differently each time.

In [15]:
comparison_table = pd.DataFrame({
    "fixed_split_f1": f1_table_42["f1-score"],
    "random_split_f1": f1_table_random["f1-score"]
})

comparison_table

,fixed_split_f1,random_split_f1
benign,0.972678,0.972678
malignant,0.948454,0.948454
accuracy,0.964286,0.964286
macro avg,0.960566,0.960566
weighted avg,0.964372,0.964372


## Cell 16 — Optional: plot training history

This is optional but useful for CNNs.

It shows how the model's accuracy changed during training.

In [16]:
import matplotlib.pyplot as plt

plt.plot(history_42.history["accuracy"], label="training accuracy")
plt.plot(history_42.history["val_accuracy"], label="validation accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fixed split CNN training history")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'